### Cleaning the Data

In [ ]:

# Imports & parameters
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

# Default path (change if needed)
file_path = 'Data for technical assessment.xlsx'
out_dir = Path('data/cleaned_results')
out_dir.mkdir(exist_ok=True)

# Display settings
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:

def drop_all_zero_rows(df):
    """Drop rows where ALL numeric columns are 0 or NaN."""
    # Copy to avoid modifying original
    df2 = df.copy()
    # coerce all non-first columns to numeric if possible, leaving first as-is
    numeric = df2.columns[1:]
    for c in numeric:
        df2[c] = pd.to_numeric(df2[c], errors='coerce')
    # Determine rows where all numeric columns are either 0 or NaN
    mask_all_zero = df2[numeric].fillna(0).abs().sum(axis=1) == 0
    return df2.loc[~mask_all_zero].reset_index(drop=True)

def tidy_sheet_with_merged_headers(df, sheet_name):
    """Convert sheet where headers are already merged (metric + year in single row).
    Returns long tidy DataFrame with columns: Firm, year, metric, value, sheet."""
    # df already has proper headers, first column is 'Firm'
    df_data = df.copy().reset_index(drop=True)

    # Parse each column (except Firm) to extract metric and year
    records = []
    for _, row in df_data.iterrows():
        firm = row['Firm']
        for col in df_data.columns[1:]:
            # Column format is like "GWP (£m) 2020YE" or "Net combined ratio 2019YE"
            parts = str(col).split()
            if len(parts) >= 2:
                year_label = parts[-1]  # Last part is year
                metric = " ".join(parts[:-1])  # Everything else is metric
            else:
                metric = col
                year_label = 'unknown'

            # Clean year
            year_str = year_label.replace('YE','').strip()
            try:
                year = int(year_str)
            except:
                year = np.nan

            # Get value
            value = row[col]
            try:
                value = float(value)
            except:
                value = np.nan

            records.append({
                'Firm': str(firm).strip(),
                'year': year,
                'metric': metric.strip(),
                'value': value,
                'sheet': sheet_name
            })

    long = pd.DataFrame(records)
    long = long[long['metric'].notna() & long['Firm'].notna()]
    return long


In [ ]:

# Identify metric columns we will use (NWP, Net combined ratio (NCR), SCR, liabilities, equity)
available = pivot_all.columns.tolist()
def find_metric_contains(names):
    for name in names:
        cand = [c for c in available if name.lower() in str(c).lower()]
        if cand:
            return cand[0]
    return None

nwp_col = find_metric_contains(['NWP (£m)','NWP (', 'NWP'])
ncr_col = find_metric_contains(['Net combined ratio', 'Net combined', 'Net combined ratio (£m)','Net combined ratio'])
scr_col = find_metric_contains(['SCR (£m)','SCR ('])
liab_col = find_metric_contains(['Total liabilities','Total liabilities'])
equity_col = find_metric_contains(['Excess of assets','excess of assets','equity'])

print('Detected metric columns:')
print('NWP:', nwp_col)
print('NCR:', ncr_col)
print('SCR:', scr_col)
print('Liabilities:', liab_col)
print('Equity:', equity_col)

# Compute firm-level aggregates
agg = {}
if nwp_col: agg[nwp_col] = ['mean','std','count']
if scr_col: agg[scr_col] = ['mean','std']
if liab_col: agg[liab_col] = ['mean']
if equity_col: agg[equity_col] = ['mean']
# Clean agg dict
agg = {k:v for k,v in agg.items() if k is not None and v is not None}

firm_stats = pivot_all.groupby('Firm').agg(agg)
# flatten
firm_stats.columns = ['_'.join([str(a) for a in col if str(a)!='']).strip() for col in firm_stats.columns.values]
firm_stats = firm_stats.reset_index()

# compute NWP-derived columns
if nwp_col:
    firm_stats['NWP_mean'] = firm_stats.get(f"{nwp_col}_mean")
    firm_stats['NWP_std'] = firm_stats.get(f"{nwp_col}_std")
    firm_stats['NWP_count'] = firm_stats.get(f"{nwp_col}_count")
    firm_stats['NWP_cv'] = firm_stats['NWP_std'] / firm_stats['NWP_mean'].replace(0, np.nan)
else:
    firm_stats[['NWP_mean','NWP_std','NWP_count','NWP_cv']] = np.nan

if scr_col:
    firm_stats['SCR_mean'] = firm_stats.get(f"{scr_col}_mean")
    firm_stats['SCR_std'] = firm_stats.get(f"{scr_col}_std")
else:
    firm_stats[['SCR_mean','SCR_std']] = np.nan

if liab_col:
    firm_stats['Liabilities_mean'] = firm_stats.get(f"{liab_col}_mean")
else:
    firm_stats['Liabilities_mean'] = np.nan

if equity_col:
    firm_stats['Equity_mean'] = firm_stats.get(f"{equity_col}_mean")
else:
    firm_stats['Equity_mean'] = np.nan

# size metric and ranks
firm_stats['size_metric'] = np.log1p(firm_stats['NWP_mean'].abs().fillna(0)) + np.log1p(firm_stats['Liabilities_mean'].abs().fillna(0))
firm_stats['size_rank'] = firm_stats['size_metric'].rank(ascending=False, method='min')

# volatility rank
firm_stats['vol_rank'] = firm_stats['NWP_cv'].rank(ascending=False, method='min').fillna(firm_stats['size_rank'].max()+1)

# outlier detection (per year z-score) for NWP, SCR, NCR if present
outlier_cols = [c for c in [nwp_col, scr_col, ncr_col] if c]
pivot_for_out = pivot_all.copy()
outlier_counts = {}
for col in outlier_cols:
    pivot_for_out[f"{col}_z"] = pivot_for_out.groupby('Year')[col].transform(lambda x: stats.zscore(x, nan_policy='omit'))
    pivot_for_out[f"{col}_is_out"] = pivot_for_out[f"{col}_z"].abs() > 2
    oc = pivot_for_out.groupby('Firm')[f"{col}_is_out"].sum().rename(f"outlier_count_{col}")
    outlier_counts[col] = oc

if outlier_counts:
    outlier_df = pd.concat(list(outlier_counts.values()), axis=1).reset_index()
    if outlier_df.columns[0] != 'Firm':
        outlier_df = outlier_df.rename(columns={outlier_df.columns[0]:'Firm'})
    outlier_df = outlier_df.fillna(0)
    outlier_df['outlier_total'] = outlier_df[[c for c in outlier_df.columns if c.startswith('outlier_count_')]].sum(axis=1)
else:
    outlier_df = pd.DataFrame({'Firm': firm_stats['Firm'], 'outlier_total': 0})

summary = firm_stats.merge(outlier_df[['Firm','outlier_total']], on='Firm', how='left').fillna(0)

# normalize and compute attention score
summary['size_norm'] = (summary['size_rank'] - summary['size_rank'].min()) / (summary['size_rank'].max() - summary['size_rank'].min() + 1e-9)
summary['vol_norm'] = (summary['vol_rank'] - summary['vol_rank'].min()) / (summary['vol_rank'].max() - summary['vol_rank'].min() + 1e-9)
summary['outlier_norm'] = summary['outlier_total'] / (summary['outlier_total'].max() + 1e-9)
w_size, w_vol, w_out = 0.45, 0.35, 0.20
summary['attention_score'] = w_size*summary['size_norm'] + w_vol*summary['vol_norm'] + w_out*summary['outlier_norm']

summary = summary.sort_values('attention_score', ascending=False).reset_index(drop=True)
summary['attention_rank'] = summary['attention_score'].rank(ascending=False, method='first')

# Save outputs
pivot_all.to_csv(out_dir / 'pivot_firm_year.csv', index=False)
combined_long.to_csv(out_dir / 'combined_long.csv', index=False)
summary.to_csv(out_dir / 'firm_attention_summary.csv', index=False)

print('\nSaved cleaned files to:', out_dir)
print('\nTop 10 firms by attention score:')
display(summary[['Firm','attention_score','attention_rank','size_norm','vol_norm','outlier_norm','NWP_mean','NWP_std','NWP_cv','SCR_mean','outlier_total']].head(10))


Detected metric columns:
NWP: NWP (£m)
NCR: Net combined ratio
SCR: EoF for SCR (£m)
Liabilities: Total liabilities (£m)
Equity: Excess of assets over liabilities (£m) [= equity]

Saved cleaned files to: data/cleaned_results

Top 10 firms by attention score:


,Firm,attention_score,attention_rank,size_norm,vol_norm,outlier_norm,NWP_mean,NWP_std,NWP_cv,SCR_mean,outlier_total
0,Firm 314,0.800,1.0,1.000000,1.0,0.0,0.0,0.0,0.0,2.199530,0
1,Firm 102,0.798,2.0,0.995556,1.0,0.0,0.0,0.0,0.0,1.847256,0
2,Firm 83,0.796,3.0,0.991111,1.0,0.0,0.0,0.0,0.0,27.621874,0
3,Firm 18,0.794,4.0,0.986667,1.0,0.0,0.0,0.0,0.0,4.975884,0
4,Firm 127,0.792,5.0,0.982222,1.0,0.0,0.0,0.0,0.0,0.143107,0
5,Firm 94,0.784,6.0,0.964444,1.0,0.0,0.0,0.0,0.0,12.501844,0
6,Firm 84,0.782,7.0,0.960000,1.0,0.0,0.0,0.0,0.0,0.328723,0
7,Firm 270,0.780,8.0,0.955556,1.0,0.0,0.0,0.0,0.0,3.637270,0
8,Firm 291,0.774,9.0,0.942222,1.0,0.0,0.0,0.0,0.0,6.742853,0
9,Firm 217,0.772,10.0,0.937778,1.0,0.0,0.0,0.0,0.0,68.516862,0


In [ ]:
# import pandas as pd

# # Load and clean both sheets
# file_path = "Data for technical assessment.xlsx"
# sheet1_raw = pd.read_excel(file_path, sheet_name="Dataset 1 - General", engine="openpyxl", header=None)
# sheet2_raw = pd.read_excel(file_path, sheet_name="Dataset 2 - Underwriting", engine="openpyxl", header=None)

# # Merge first two rows into header
# header1 = [f"{a} {b}".strip() for a, b in zip(sheet1_raw.iloc[0], sheet1_raw.iloc[1])]
# sheet1 = sheet1_raw.iloc[2:].copy()
# sheet1.columns = header1
# sheet1.rename(columns={sheet1.columns[0]: "Firm"}, inplace=True)

# header2 = [f"{a} {b}".strip() for a, b in zip(sheet2_raw.iloc[0], sheet2_raw.iloc[1])]
# sheet2 = sheet2_raw.iloc[2:].copy()
# sheet2.columns = header2
# sheet2.rename(columns={sheet2.columns[0]: "Firm"}, inplace=True)

# # Merge sheets
# merged = pd.merge(sheet1, sheet2, on="Firm", how="inner")

In [ ]:
# # Extract key metrics
# metrics = ["GWP (£m)", "NWP (£m)", "SCR coverage ratio", "Gross claims incurred (£m)", "Net combined ratio"]
# metric_cols = [col for col in merged.columns if any(m in col for m in metrics)]

# # Tidy dataframe
# records = []
# for _, row in merged.iterrows():
#     firm = row['Firm']
#     for col in metric_cols:
#         parts = col.split()
#         metric = " ".join(parts[:-1])
#         year = parts[-1]
#         try:
#             value = float(row[col])
#         except:
#             value = None
#         records.append({"Firm": firm, "Year": year, "Metric": metric, "Value": value})

# tidy_df = pd.DataFrame(records)

In [20]:
#some data sense checks
# for each year, check if NWP <= GWP for each firm


### Working with Cleaned Data

In [1]:
import pandas as pd
tidy_df_updated = pd.read_csv('data/cleaned_results/combined_long.csv')

In [2]:
# YoY change table
# Ensure correct ordering
tidy_df_updated = tidy_df_updated.sort_values(by=['Firm', 'Metric', 'Year'])

# Calculate YoY change by firm and metric
tidy_df_updated['YoY_change'] = tidy_df_updated.groupby(['Firm', 'Metric'])['Value'].pct_change()

In [3]:
# Summary for prioritization
summary = tidy_df_updated.groupby(['Firm', 'Metric']).agg({"Value": ["mean", "std"]}).reset_index()
summary.columns = ['Firm', 'Metric', 'AvgValue', 'StdDev']

# Identify top firms and outliers
largest_firms = summary[summary['Metric'] == 'GWP (£m)'].sort_values('AvgValue', ascending=False).head(10)
volatile_firms = summary.sort_values('StdDev', ascending=False).head(10) # pick which metric we care about here
outliers_scr = summary[(summary['Metric'] == 'SCR coverage ratio') & (summary['AvgValue'] < 1)].sort_values('AvgValue').head(10)
outliers_combined = summary[(summary['Metric'] == 'Net combined ratio') & (summary['AvgValue'] > 1)].sort_values('AvgValue', ascending=False).head(10)

In [16]:
summary

,Firm,Metric,AvgValue,StdDev
0,Firm 1,EoF for SCR (£m),446.978924,996.784143
1,Firm 1,Excess of assets over liabilities (£m) [= equity],407.170771,907.770621
2,Firm 1,GWP (£m),281.896959,630.340763
3,Firm 1,"Gross BEL (inc. TPs as whole, pre-TMTP) (£m)",1.534894,3.432127
4,Firm 1,Gross claims incurred (£m),0.009335,0.020873
...,...,...,...,...
3837,Firm 99,Pure net claims ratio,-7470.127880,16703.713740
3838,Firm 99,SCR (£m),257.514827,14.817735
3839,Firm 99,SCR coverage ratio,1.491485,0.124202
3840,Firm 99,Total assets (£m),1044.942180,139.034947


In [4]:
import plotly.express as px

# Create charts
fig_gwp = px.bar(largest_firms, x='Firm', y='AvgValue', title='Top 10 Firms by GWP (£m)')
# fig_gwp.write_image('largest_firms.png')
# fig_gwp.write_json('largest_firms.json')

fig_vol = px.bar(volatile_firms, x='Firm', y='StdDev', title='Top 10 Most Volatile Firms')
# fig_vol.write_image('volatile_firms.png')
# fig_vol.write_json('volatile_firms.json')

fig_scr = px.bar(outliers_scr, x='Firm', y='AvgValue', title='Low SCR Coverage Ratio (<1)')
# fig_scr.write_image('low_scr.png')
# fig_scr.write_json('low_scr.json')

fig_combined = px.bar(outliers_combined, x='Firm', y='AvgValue', title='High Net Combined Ratio (>1)')
# fig_combined.write_image('high_combined.png')
# fig_combined.write_json('high_combined.json')


In [5]:
fig_gwp.show()
fig_vol.show()
fig_scr.show()
fig_combined.show()

In [21]:
df_combined = tidy_df_updated[tidy_df_updated["Metric"] == "Net combined ratio"]
df_nwp = tidy_df_updated[tidy_df_updated["Metric"] == "NWP (£m)"]
df_gwp = tidy_df_updated[tidy_df_updated["Metric"] == "GWP (£m)"]

df_plot = pd.merge(df_combined, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
df_plot = pd.merge(df_plot, df_gwp, on=["Firm", "Year"], suffixes=("", "_gwp"))

fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)


In [22]:
fig_combined_vs_nwp.show()

In [23]:
# plotting the same as above, restricting net combined ratios up to 1000 and >= 0

df_combined_restricted = tidy_df_updated[
    (tidy_df_updated["Metric"] == "Net combined ratio") &
    (tidy_df_updated["Value"] <= 1000) &
    (tidy_df_updated["Value"] >= -1000)
]

df_nwp = tidy_df_updated[tidy_df_updated["Metric"] == "NWP (£m)"]

df_plot = pd.merge(df_combined_restricted, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))

fig_combined_vs_nwp_restricted = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium - outliers removed",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)

In [24]:
fig_combined_vs_nwp_restricted.show()

In [ ]:
#plot NWP vs GWP for 2020
fig_gwp_vs_nwp = px.scatter(
    df_plot[df_plot["Year"] == '2020YE'],
    x="Value",
    y="Value_nwp",
    title="Gross Written Premium vs Net Written Premium for 2020",
    labels={"Value": "Gross Written Premium (£m)", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm"]
)


In [ ]:
fig_gwp_vs_nwp.show()

### Using only 2020 data

In [90]:
import pandas as pd
tidy_df_updated = pd.read_csv('data/cleaned_results/combined_long.csv')

In [91]:
# YoY change table
# Ensure correct ordering
tidy_df_updated = tidy_df_updated.sort_values(by=['Firm', 'Metric', 'Year'])

# Calculate YoY change by firm and metric
tidy_df_updated['YoY_change'] = tidy_df_updated.groupby(['Firm', 'Metric'])['Value'].pct_change()

In [92]:
tidy_df_updated_2020 = tidy_df_updated[tidy_df_updated["Year"]==2020]
tidy_df_updated_2020

,Firm,Year,Metric,Value,YoY_change
14,Firm 1,2020,EoF for SCR (£m),0.000000,NaN
39,Firm 1,2020,Excess of assets over liabilities (£m) [= equity],0.000000,NaN
24,Firm 1,2020,GWP (£m),0.000000,NaN
49,Firm 1,2020,"Gross BEL (inc. TPs as whole, pre-TMTP) (£m)",0.000000,NaN
44,Firm 1,2020,Gross claims incurred (£m),0.000000,NaN
...,...,...,...,...,...
6009,Firm 99,2020,Pure net claims ratio,0.000000,NaN
5959,Firm 99,2020,SCR (£m),281.473991,0.146232
5969,Firm 99,2020,SCR coverage ratio,1.378122,-0.166187
5979,Firm 99,2020,Total assets (£m),1215.246388,0.086642


#### Visualise selected metrics (Top 10 firms)

In [72]:
# Identify top firms and outliers
largest_firms = tidy_df_updated_2020[tidy_df_updated_2020['Metric'] == 'GWP (£m)'].sort_values('Value', ascending=False).head(10)
# volatile_firms = tidy_df_updated_2020.sort_values('StdDev', ascending=False).head(10) # pick which metric we care about here
outliers_scr = tidy_df_updated_2020[(tidy_df_updated_2020['Metric'] == 'SCR coverage ratio') & (tidy_df_updated_2020['Value'] < 1)].sort_values('Value').head(10)
outliers_combined = tidy_df_updated_2020[(tidy_df_updated_2020['Metric'] == 'Net combined ratio') & (tidy_df_updated_2020['Value'] > 1)].sort_values('Value', ascending=False).head(10)

In [73]:
tidy_df_updated_2020[tidy_df_updated_2020["Metric"]=="SCR coverage ratio"]

,Firm,Year,Metric,Value,YoY_change
19,Firm 1,2020,SCR coverage ratio,0.000000,NaN
529,Firm 10,2020,SCR coverage ratio,1.421303,-0.147538
6054,Firm 100,2020,SCR coverage ratio,1.077108,0.008382
6139,Firm 102,2020,SCR coverage ratio,0.000000,NaN
6224,Firm 104,2020,SCR coverage ratio,8.080187,1.577630
...,...,...,...,...,...
5629,Firm 92,2020,SCR coverage ratio,1.946785,-0.213788
5714,Firm 94,2020,SCR coverage ratio,13.616788,0.002486
5799,Firm 96,2020,SCR coverage ratio,0.000000,NaN
5884,Firm 97,2020,SCR coverage ratio,1.475222,-0.459535


In [74]:
import plotly.express as px

# Create charts
fig_gwp = px.bar(largest_firms, x='Firm', y='Value', title='Top 10 Firms by GWP (£m) in 2020')
# fig_gwp.write_image('largest_firms.png')
# fig_gwp.write_json('largest_firms.json')

# fig_vol = px.bar(volatile_firms, x='Firm', y='StdDev', title='Top 10 Most Volatile Firms')
# fig_vol.write_image('volatile_firms.png')
# fig_vol.write_json('volatile_firms.json')

fig_scr = px.bar(outliers_scr, x='Firm', y='Value', title='Low SCR Coverage Ratio (<1) in 2020')
# fig_scr.write_image('low_scr.png')
# fig_scr.write_json('low_scr.json')

fig_combined = px.bar(outliers_combined, x='Firm', y='Value', title='High Net Combined Ratio (>1) in 2020')
# fig_combined.write_image('high_combined.png')
# fig_combined.write_json('high_combined.json')

In [76]:
fig_gwp.show()
# fig_vol.show()
fig_scr.show()
fig_combined.show()

#### Visualise NWP vs. GWP

In [87]:
df_combined = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "Net combined ratio"]
df_nwp = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "NWP (£m)"]
df_gwp = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "GWP (£m)"]

df_plot = pd.merge(df_combined, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
df_plot = pd.merge(df_plot, df_gwp, on=["Firm", "Year"], suffixes=("", "_gwp"))

fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium in 2020",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"]
)
fig_combined_vs_nwp.show()

In [ ]:
import plotly.express as px

# Create a new column for text labels, only if condition is met
# Add conditional text labels
df_plot['label'] = df_plot.apply(
    lambda row: f"{row['Firm']}"
                if (row['Value_combined'] > 40 or row['Value_nwp'] > 20000)
                else '',
    axis=1
)

# Create the scatter plot
fig_combined_vs_nwp = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium in 2020",
    labels={
        "Value_combined": "Net Combined Ratio",
        "Value_nwp": "Net Written Premium (£m)"
    },
    hover_data=["Firm", "Year"],
    text="label"  # Add labels
)

# Position text next to the points
fig_combined_vs_nwp.update_traces(textposition="top center")
fig_combined_vs_nwp.show()


In [ ]:
# plotting the same as above, restricting net combined ratios up to 200 and >= 0

df_combined_restricted = tidy_df_updated_2020[
    (tidy_df_updated_2020["Metric"] == "Net combined ratio") &
    (tidy_df_updated_2020["Value"] <= 200) &
    (tidy_df_updated_2020["Value"] >= -200)
]

df_nwp = tidy_df_updated_2020[tidy_df_updated_2020["Metric"] == "NWP (£m)"]

df_plot = pd.merge(df_combined_restricted, df_nwp, on=["Firm", "Year"], suffixes=("_combined", "_nwp"))
# Add conditional text labels
df_plot['label'] = df_plot.apply(
    lambda row: f"{row['Firm']}"
                if (row['Value_combined'] > 10 or row['Value_nwp'] > 20000)
                else '',
    axis=1
)

fig_combined_vs_nwp_restricted = px.scatter(
    df_plot,
    x="Value_combined",
    y="Value_nwp",
    title="Net Combined Ratio vs Net Written Premium in 2020- outliers removed",
    labels={"Value_combined": "Net Combined Ratio", "Value_nwp": "Net Written Premium (£m)"},
    hover_data=["Firm", "Year"],
    text="label"  # Add labels
)

# Position text next to the points
fig_combined_vs_nwp_restricted.update_traces(textposition="top center")
fig_combined_vs_nwp_restricted.show()

#### Visualise YoY change for all metrics

In [93]:
# plot the most volitile (YoY change for 2020-2019)
# missing value in YoY change means both current and previour year have value 0
# -1 in YoY change means current year has value 0 while preious year doesn't
for i in tidy_df_updated_2020["Metric"].unique():
    temp_df = tidy_df_updated_2020[tidy_df_updated_2020["Metric"]==i].sort_values(by="YoY_change", ascending=False).head(10)
    fig_YoY = px.bar(temp_df, x='Firm', y='YoY_change', title=f'Top 10 Firms by 2019 - 2020 YoY change for "{i}" ')
    fig_YoY.show()